# 🦖 Example 6: Deep-Time Paleo-Globes, 4-Filament Mixing & Interactive 3D Preview

This tutorial explains how to create physical, 3D printable globes of ancient Earth across deep geological time (0 to 540 Ma). It covers plate tectonic reconstructions, Scotese PALEOMAP PaleoDEM elevation grids, physics-based full-spectrum filament mixing using 4 base filaments (Brown, Dark Blue, White, Dark Green) for multi-material FDM 3D printing, discrete banded colormaps matching the Scotese hypsometric palette, and interactive 3D WebGL rendering with `model.preview()`.

---

### 🌎 Key Capabilities Demonstrated:
1. **Paleo Datasets & Catalog**: Querying 108 geological time slices from the Cambrian (540 Ma) to modern day via `globe3d.datasets.paleo`.
2. **Deep-Time Coordinate Tracking**: Tracing modern landmarks (Oxford, UK and Baltimore, USA) across continental drift to the Permo-Triassic supercontinent **Pangaea** (250 Ma) using `PlateModel`.
3. **Full-Spectrum Optical Filament Mixing**: Predicting perceived optical layer mixing colors from physical base spools using a Kubelka-Munk polynomial model (as per *Full Spectrum 3D* / *OrcaSlicer-FullSpectrum* / *PolyMixer*).
4. **4 Base Filaments Palette**: Configuring a 4-spool system (**Brown**, **Dark Blue**, **White**, **Dark Green**) suited for standard 4-slot filament systems (e.g. Bambu AMS).
5. **Scotese-Matched Hypsometric Colormaps**: Constructing discrete 8-zone elevation colormaps with refined geological thresholds that accurately reflect ancient continental shelves, savannas, plateaus, and snow-capped peaks.
6. **Texture Preprocessing & Stamping**: Eliminating hillshade baked shadows, stamping spherical SVG markers at reconstructed coordinates, and adding North/South polar markers.
7. **3D Displaced Globe Generation**: Building hollowed, vertically exaggerated spherical models with coastline shelf steps via `GlobeModel.from_paleo`.
8. **Interactive 3D WebGL Preview**: Orbiting, panning, and zooming the textured 3D globe directly inside the notebook using `model.preview()`.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import globe3d
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    ImageColourer,
    PlateModel,
    datasets,
    mix_filament_pair,
    mix_filaments,
    create_filament_palette,
    match_filament_color,
    create_banded_colormap,
    create_elevation_banded_colormap,
    apply_banded_colormap,
    plot_filament_palette,
    plot_banded_colormap_preview,
    undo_hillshading,
    add_svg_symbol_spherical,
    add_pole_markers,
)

print(f"globe3d version: {getattr(globe3d, '__version__', '0.1.0')}")

## 1. Inspecting the Scotese PALEOMAP Catalog
`globe3d` includes a registry of 108 reconstructed time slices based on Scotese & Wright (2018) PALEOMAP PaleoDEMs and matching PaleoAtlas textures.

In [ ]:
maps = datasets.paleo.list_scotese_maps()
print(f"Total available geological time slices: {len(maps)}")

print("\nSample time slices:")
for m in maps[::12]:  # Sample every 12th map
    print(f"  {m['age_ma']:>5.1f} Ma: Map {m['map_id']:02d} — {m['name']}")

## 2. Plate Tectonic Reconstructions: Tracing Modern Cities
Using the PALEOMAP Global Plate Model, we can trace where modern cities were situated millions of years ago.
Let's track **Oxford, UK** (51.75°N, -1.26°E) and **Baltimore, USA** (39.29°N, -76.61°W) at the **Permo-Triassic Boundary (250 Ma)**, when all major landmasses were merged into the supercontinent **Pangaea**:

In [ ]:
pm = datasets.paleo.scotese_plate_model()

oxford_modern = [51.7519, -1.2578]
baltimore_modern = [39.2904, -76.6122]

oxford_250ma = pm.reconstruct_points(oxford_modern, age_ma=250.0)[0]
baltimore_250ma = pm.reconstruct_points(baltimore_modern, age_ma=250.0)[0]

print("Oxford, UK:")
print(f"  Modern:         {oxford_modern[0]:.2f}°N, {oxford_modern[1]:.2f}°E")
print(f"  250 Ma Pangaea: {oxford_250ma[0]:.2f}°N, {oxford_250ma[1]:.2f}°E (Subtropical Pangaean interior)\n")

print("Baltimore, USA:")
print(f"  Modern:         {baltimore_modern[0]:.2f}°N, {baltimore_modern[1]:.2f}°E")
print(f"  250 Ma Pangaea: {baltimore_250ma[0]:.2f}°N, {baltimore_250ma[1]:.2f}°E (Central Pangaean suture)")

## 3. Full-Spectrum Optical Filament Mixing for Multi-Material 3D Printing

In multi-material FDM 3D printing (e.g. Bambu AMS, Prusa XL / MMU, or tool-changers), alternating ultra-thin layers (0.08–0.12 mm) of semi-translucent base filaments causes light to pass through adjacent layers, producing a perceived blend of color (optical mixing / pointillism).

### Why Naive RGB Averaging Fails:
Standard slicers that compute colors with naive linear RGB interpolation `(1-t)*C1 + t*C2` fail to model light scattering and absorption:
- Mixing Blue `(18, 52, 120)` and Yellow naively gives a **muddy desaturated brown/gray**.
- In physical reality (and under Kubelka-Munk pigment physics), blue absorbs red/yellow light while yellow absorbs blue, allowing green light to reflect — producing **vibrant green**.

`globe3d.filaments` includes a learned degree-4 Kubelka-Munk polynomial model (identical to *Full Spectrum 3D* / *PolyMixer*) that accurately predicts perceived layered blends:

In [ ]:
blue = (0, 33, 133)
yellow = (252, 211, 0)

# Naive linear RGB mix
naive_green = tuple(int(0.5 * b + 0.5 * y) for b, y in zip(blue, yellow))

# Full Spectrum Kubelka-Munk polynomial mix
km_green = mix_filament_pair(blue, yellow, ratio=0.5)

print(f"Blue:           {blue}")
print(f"Yellow:         {yellow}\n")
print(f"Naive RGB mix:  {naive_green} -> Muddy Brown/Gray")
print(f"Full-Spectrum:  {km_green} -> Vibrant Green (47, 141, 56)!")

## 4. The 4 Base Filament System: Brown, Dark Blue, White, Dark Green

Rather than requiring 5 process color spools (CMYKW), planetary and topographic globes can be printed with 4 dedicated physical base spools loaded in a standard 4-slot filament system (e.g. Bambu AMS):
1. **Brown** (`(142, 68, 28)` / `#8e441c`): Mountain ranges, plateaus, and continental suture belts.
2. **Dark Blue** (`(18, 52, 120)` / `#123478`): Deep oceanic trenches and abyssal Panthalassa plains.
3. **White** (`(250, 250, 250)` / `#fafafa`): High transmission distance (`td=1.5`), creates continental shelf tints and snow-capped alpine peaks.
4. **Dark Green** (`(32, 95, 42)` / `#205f2a`): Coastal lowlands, river valleys, and transitional shelves.

`create_filament_palette(filaments_4)` calculates all pairwise blends, tints (with White), and multi-layer combinations across these 4 spools:

In [ ]:
# Inspect the 4 base physical filament spools
filaments_4 = [
    {"name": "Brown", "color": (142, 68, 28), "td": 0.8},
    {"name": "Dark Blue", "color": (18, 52, 120), "td": 1.0},
    {"name": "White", "color": (250, 250, 250), "td": 1.5},
    {"name": "Dark Green", "color": (32, 95, 42), "td": 0.9},
]

print("4 Base Filaments:")
for f in filaments_4:
    hex_code = f"#{f['color'][0]:02x}{f['color'][1]:02x}{f['color'][2]:02x}"
    print(f"  {f['name']:<12} RGB: {str(f['color']):<16} Hex: {hex_code}  td: {f['td']} mm")

# Generate virtual printable palette across the 4 base spools
# (Accepts either the explicit list of spools above, or the built-in preset name "TERRAIN_4")
palette_4 = create_filament_palette(filaments_4, ratios=[0.25, 0.5, 0.75], include_tints=True, include_shades=False)
print(f"\nGenerated {len(palette_4)} printable optical color mixtures from the 4 base spools!")

# Display swatch preview of mixed palette
fig, ax = plt.subplots(figsize=(14, 3.5))
plot_filament_palette(palette_4[:20], title="4-Filament Virtual Palette (Brown, Dark Blue, White, Dark Green)", ax=ax)
plt.show()

## 5. Setting Up Discrete Banded Colormaps for 3D Printing

Physical 3D printers cannot print continuous analog color gradients without dithering. To print an elevation model, the continuous topography must be partitioned into **discrete elevation bands**, where each band is assigned a specific mixed filament recipe.

`create_elevation_banded_colormap('TERRAIN_4')` partitions the continuous elevation data into **8 discrete geological zones**, matched to the hypsometric distribution of ancient continents and oceans:

| Band | Elevation Zone | Range (m) | Optical Recipe | Resulting Color |
|---|---|---|---|---|
| 1 | Abyssal Plain | < -1,500 m | 100% Dark Blue | `#123478` (Abyssal Navy) |
| 2 | Continental Slope | -1,500 to -250 m | 65% Dark Blue + 15% Dark Green + 20% White | `#2c6b94` (Slope Steel Blue) |
| 3 | Continental Shelf & Shallow Seas | -250 to 0 m | 30% Dark Blue + 10% Dark Green + 60% White | `#85b7cc` (Shelf Cyan Blue) |
| 4 | Coastal Lowlands | 0 to 150 m | 75% Dark Green + 15% Brown + 10% White | `#477135` (Emerald Forest Green) |
| 5 | Interior Savanna & Plains | 150 to 450 m | 40% Dark Green + 35% Brown + 25% White | `#817d4f` (Savanna Golden-Olive) |
| 6 | Midlands & Plateaus | 450 to 950 m | 55% Brown + 25% White + 20% Dark Green | `#97734c` (Upland Golden Ochre) |
| 7 | Mountain Ranges | 950 to 1,500 m | 85% Brown + 15% White | `#a55933` (Terracotta Mountain Ridge) |
| 8 | Alpine High Peaks | > 1,500 m | 100% White | `#fafafa` (Glacier Snow White) |

In [ ]:
# Generate the 8 discrete elevation bands from the 4 base filaments
cmap, norm, band_specs = create_elevation_banded_colormap(filaments_4)

print("4-Filament Geological Elevation Bands & Physical Recipes (Scotese Match):")
print("-" * 80)
for idx, b in enumerate(band_specs, 1):
    recipe_str = " + ".join(f"{int(w*100)}% {name}" for name, w in b['recipe'])
    print(f"Band {idx}: {b['name']:<34} {b['range']:<18} Hex: {b['hex']}  Recipe: {recipe_str}")

# Preview the discrete banded colorbar
fig, ax = plt.subplots(figsize=(13, 2.0))
thresholds = [-1500, -250, 0, 150, 450, 950, 1500]
band_names = [b['name'] for b in band_specs]
plot_banded_colormap_preview(cmap, norm, thresholds, band_labels=band_names, title="Geological Elevation Bands (4 Base Filaments)", ax=ax)
plt.show()

## 6. 2D Texture Preprocessing & Rendering

Before wrapping a texture onto a 3D globe, we preprocess it for 3D printing:
1. **Apply Banded Colormap**: Transform the continuous 250 Ma PaleoDEM elevation array into discrete filament color bands using `apply_banded_colormap`.
2. **Undo Hillshading**: Remove synthetic lighting shadows baked into the Scotese PaleoAtlas map texture.
3. **Spherical Vector Stamping**: Stamp reconstructed Oxford and Baltimore star logos, plus North/South pole markers.

In [ ]:
# Load 250 Ma PaleoDEM elevation grid and texture from datasets
dem_grid, texture_path, meta = datasets.paleo.scotese(250.0)
raw_texture = Image.open(texture_path)
print(f"Loaded: {meta['name']} ({meta['age_ma']} Ma)")
print(f"PaleoDEM Grid shape: {dem_grid.data.shape}, range: [{dem_grid.data.min():.0f}m, {dem_grid.data.max():.0f}m]")

# 1. Generate discrete 8-band texture using the 4-filament palette
band_colors = [b['color'] for b in band_specs]
banded_texture_arr = apply_banded_colormap(dem_grid.data, band_colors, thresholds)
banded_img = Image.fromarray(banded_texture_arr)

# 2. Preprocess raw texture: remove hillshading
delit_img = undo_hillshading(raw_texture, strength=0.6)

# 3. Stamp reconstructed location markers and pole markers
marked_img = add_svg_symbol_spherical(delit_img, "star", oxford_250ma[0], oxford_250ma[1], symbol_radius_degrees=3.0, fill_color=(255, 0, 0))
marked_img = add_svg_symbol_spherical(marked_img, "star", baltimore_250ma[0], baltimore_250ma[1], symbol_radius_degrees=3.0, fill_color=(255, 255, 0))
marked_img = add_pole_markers(marked_img, marker_size_degrees=2.5, north_symbol="N", south_symbol="S")

# 2D Comparison Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(raw_texture)
axes[0].set_title("1. Raw Scotese PaleoAtlas (250 Ma)", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(marked_img)
axes[1].set_title("2. Hillshade-Removed + SVG Star & Pole Glyphs", fontweight="bold")
axes[1].axis("off")

axes[2].imshow(banded_img)
axes[2].set_title("3. Banded Texture (4 Filaments: Brown, Dark Blue, White, Dark Green)", fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 7. Creating the 3D Displaced Globe Model

We construct the 3D spherical mesh with `GlobeModel.from_paleo`:
- `radius=100.0`: 100 mm radius (200 mm diameter globe).
- `n_points=15000`: Mesh vertex density for fast, smooth rendering.
- `coastline_step=1000.0`: 1000 m synthetic shelf vertical step to accentuate ancient shorelines.
- `vert_exagg=35.0`: 35x vertical exaggeration so mountains and deep ocean trenches are physically tactile.
- `texture=banded_img`: Applies our 4-filament discrete banded hypsometric texture.
- `tracked_locations`: Automatically burns Oxford and Baltimore star glyphs onto their reconstructed 250 Ma positions.
- `mark_poles=True`: Automatically burns North ('N') and South ('S') polar markers onto the texture.
- `hollow=True`: Hollows out the sphere interior to conserve filament.

In [ ]:
model = GlobeModel.from_paleo(
    age=250.0,
    radius=100.0,
    n_points=15000,
    hollow=True,
    inner_ratio=0.9,
    vert_exagg=35.0,
    coastline_step=1000.0,
    texture=banded_img,
    mark_poles=True,
    tracked_locations=[
        {"lat": 51.7519, "lon": -1.2578, "name": "Oxford", "symbol": "star", "size": 3.0, "color": (255, 0, 0)},
        {"lat": 39.2904, "lon": -76.6122, "name": "Baltimore", "symbol": "star", "size": 3.0, "color": (255, 255, 0)},
    ]
)

print("Created 250 Ma Pangaea GlobeModel with 4-Filament Banded Texture!")
print(f"Outer mesh vertices: {len(model.outer.vertices):,}")
print(f"Inner mesh vertices: {len(model.inner.vertices):,}")
print(f"Radial span: [{np.linalg.norm(model.outer.vertices, axis=1).min():.1f} mm, {np.linalg.norm(model.outer.vertices, axis=1).max():.1f} mm]")

## 8. Interactive 3D WebGL Mesh Preview with `model.preview()`

Calling `model.preview()` embeds an interactive 3D WebGL Three.js viewer directly in the notebook:
- **Orbit / Rotate**: Click and drag with the left mouse button to rotate around Pangaea and view the ancient globe from any angle.
- **Pan**: Right-click (or Shift + click) and drag to pan the camera.
- **Zoom**: Scroll wheel to inspect local topography, shelf steps, and reconstructed city markers.

Let's preview the complete 3D globe with the 4-filament banded colors:

In [ ]:
# Launch interactive 3D WebGL preview of the whole globe
model.preview()

You can also preview individual hemispheres (e.g. Northern hemisphere capped at the equator) using `part='upper'` to verify the hollowed interior and wall thickness:

In [ ]:
# Preview the Northern hemisphere
model.preview(part="upper")

## 9. Magnet Alignment Voids & 3D Print Export

For printing on a standard desktop 3D printer:
1. `configure_magnets()` inserts recessed cylindrical pockets along the equator with alignment bosses.
2. `export_hemispheres()` splits the model along the equator into Northern and Southern hemispheres (with flat equatorial bases for bed adhesion).

In [ ]:
# Configure magnet cavities (e.g. 4 neodymium magnets of 6x3mm along the equator)
# model.configure_magnets(magnet_radius=3.1, magnet_thickness=3.2, count=4)

# Export hemispheres ready for slicer import
# model.export_hemispheres(
#     upper_path="paleo_250ma_north.obj",
#     lower_path="paleo_250ma_south.obj",
#     flip_lower=True,
# )

print("Paleo-globe pipeline complete and ready for physical printing!")